# Feature extraction

**Research objective.** Explain and inspect actual sparse, image, and frozen-embedding feature configurations.

All results are read from local FNEWS artifacts. `SOURCE_TRUE` and `SOURCE_NON_TRUE` are Fakeddit source-taxonomy labels, not independently verified factuality labels. The held-out test partition is not opened by these notebooks except where a prior frozen metric is explicitly quoted.

In [1]:
from pathlib import Path
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image

warnings.filterwarnings("ignore", category=FutureWarning)
ROOT = Path.cwd()
if not (ROOT / "data/research_dataset").exists():
    ROOT = Path("/Users/aka_ank/Desktop/FNEWS")
DATA = ROOT / "data/research_dataset"
REPORTS = ROOT / "reports"
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.titlesize": 13, "axes.labelsize": 11})

def paired(split):
    frame = pd.read_parquet(DATA / "manifests" / f"{split}_manifest.parquet")
    return frame.loc[(frame.source_dataset == "Fakeddit") & (frame.download_status == "verified_reused_or_processed")].copy().reset_index(drop=True)

def canonical_source_labels(raw):
    """Fakeddit raw 1=true; return 0=SOURCE_TRUE, 1=SOURCE_NON_TRUE."""
    return 1 - np.asarray(raw, dtype=int)

def source_names(values):
    return np.where(np.asarray(values) == 0, "SOURCE_TRUE", "SOURCE_NON_TRUE")

def show_frame(frame, rows=10):
    display(frame.head(rows))


## TF-IDF sparse features

For document *d* and term *t*, TF-IDF combines term frequency with inverse document frequency. The saved SVM vectorizer uses at most 50,000 unigram/bigram terms, L2 normalization, `min_df=2`, and sublinear term frequency.

In [2]:
val=paired('validate');svm_pipe=joblib.load(REPORTS/'experiments/research_dataset_baselines/tfidf_svm/model.joblib');vec=svm_pipe.named_steps['tfidf'];matrix=vec.transform(val.text);print('Validation sparse matrix:',matrix.shape,'density:',matrix.nnz/(matrix.shape[0]*matrix.shape[1]));print('Vocabulary features:',len(vec.get_feature_names_out()))

Validation sparse matrix: (933, 2964) density: 0.0019009825660697213
Vocabulary features: 2964


In [3]:
coef=svm_pipe.named_steps['svm'].coef_[0];terms=vec.get_feature_names_out();
for title,idx in [('Raw class 1 / SOURCE_TRUE',coef.argsort()[-15:][::-1]),('Raw class 0 / SOURCE_NON_TRUE',coef.argsort()[:15])]:
 print('\n'+title);display(pd.DataFrame({'term':terms[idx],'weight':coef[idx]}))


Raw class 1 / SOURCE_TRUE


,term,weight
0,this,1.905000
1,home,1.690830
2,cat,1.636336
3,with,1.550271
4,oc,1.534850
5,from,1.521975
6,license,1.497426
7,under,1.477965
8,dog,1.463305
9,two,1.454931



Raw class 0 / SOURCE_NON_TRUE


,term,weight
0,me,-2.175395
1,like this,-1.393943
2,mom,-1.264414
3,poster,-1.184022
4,soviet,-1.181205
5,strong,-1.158718
6,this horse,-1.144161
7,soldier,-1.142975
8,theres,-1.132667
9,circa,-1.130196


## Frozen text, image, and CLIP representations

The saved pretrained-text baseline is a frozen encoder baseline; its stored validation predictions are used rather than recomputing transformer embeddings. ResNet-18 uses ImageNet preprocessing and a two-class head. CLIP ViT-B/32 contributes 512-dimensional image and text embeddings to the archived gated head.

In [4]:
checkpoint_path=REPORTS/'multimodal_fusion/clip_fusion_v1/gated/best_model.pt'
import torch
checkpoint=torch.load(checkpoint_path,map_location='cpu',weights_only=False);print('CLIP fusion checkpoint config:',checkpoint['config']);print('Stored feature mean shape:',tuple(checkpoint['feature_mean'].shape));print('Stored feature std shape:',tuple(checkpoint['feature_std'].shape));print('Expected concatenation: image 512 + text 512 = 1024')

CLIP fusion checkpoint config: {'dataset_root': 'data/research_dataset', 'output_dir': 'reports/multimodal_fusion/clip_fusion_v1', 'config_copy_path': 'reports/multimodal_fusion/clip_fusion_v1/config.json', 'clip_model': 'openai/clip-vit-base-patch32', 'seed': 42, 'hidden_dim': 256, 'dropout': 0.2, 'batch_size': 64, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'epochs': 20, 'patience': 4}
Stored feature mean shape: (1024,)
Stored feature std shape: (1024,)
Expected concatenation: image 512 + text 512 = 1024


In [5]:
from torchvision.models import ResNet18_Weights
row=val.iloc[0];pixels=ResNet18_Weights.DEFAULT.transforms()(Image.open(DATA/row.image_path).convert('RGB'));print('Real validation ResNet tensor:',tuple(pixels.unsqueeze(0).shape));print('No supervised dimensionality reduction is fitted on validation labels.')

Real validation ResNet tensor: (1, 3, 224, 224)
No supervised dimensionality reduction is fitted on validation labels.


## Reproduction note

Embedding caches are not required for this notebook. Shapes are read from the preserved checkpoint and an actual bounded validation input is transformed with the recorded image pipeline.